# 🧠 GoalParser — FLAN-T5-Small Training

Trains FLAN-T5-Small to convert natural language browser instructions into structured JSON goal objects.

**Before running:** Upload `goal_parser_train.jsonl` to this Colab session using the file explorer (📁).

**Runtime:** Go to `Runtime → Change runtime type → T4 GPU`

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece protobuf

## Cell 2 — Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Edit these as needed
# ============================================================

DATA_FILE = 'goal_parser_train.jsonl'  # Your labeled dataset
MODEL_NAME = 'google/flan-t5-small'    # Base model (80M params)
OUTPUT_DIR = './goal_parser_model'      # Where to save checkpoints

# Training hyperparameters
EPOCHS = 10                 # Max epochs (early stopping will cut this short)
BATCH_SIZE = 8              # Per-device batch size (8 fits T4 easily)
LEARNING_RATE = 3e-4        # Standard for T5 fine-tuning
WEIGHT_DECAY = 0.01         # L2 regularization
LABEL_SMOOTHING = 0.1       # Prevents overconfidence
WARMUP_STEPS = 100          # Gradual LR ramp-up
MAX_INPUT_LENGTH = 256      # Max tokens for input prompt
MAX_OUTPUT_LENGTH = 512     # Max tokens for JSON output

# Data split ratios
VAL_RATIO = 0.05            # 5% for validation (checked every epoch)
TEST_RATIO = 0.10           # 10% for final test (held out completely)

# Early stopping
PATIENCE = 3                # Stop if val loss doesn't improve for 3 epochs

print('✅ Configuration loaded')
print(f'   Model: {MODEL_NAME}')
print(f'   Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}')
print(f'   Label smoothing: {LABEL_SMOOTHING}, Weight decay: {WEIGHT_DECAY}')

## Cell 3 — Load & Split Dataset

In [ ]:
import json
import os

# Verify file exists
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f'❌ {DATA_FILE} not found! Upload it via the file explorer (📁).')

# Load all entries
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    all_entries = [json.loads(line) for line in f if line.strip()]

print(f'📊 Total entries loaded: {len(all_entries)}')

# Remove garbage entries
clean_entries = [
    e for e in all_entries
    if e['input'] != 'Untitled'
    and not e['input'].startswith('http')
    and len(e['input']) >= 10
]
removed = len(all_entries) - len(clean_entries)
if removed > 0:
    print(f'🗑️  Removed {removed} garbage entries')

# Shuffle with fixed seed for reproducibility
import random
random.seed(42)
random.shuffle(clean_entries)

# Split into train / val / test
n = len(clean_entries)
n_test = int(n * TEST_RATIO)
n_val = int(n * VAL_RATIO)
n_train = n - n_test - n_val

test_data = clean_entries[:n_test]
val_data = clean_entries[n_test:n_test + n_val]
train_data = clean_entries[n_test + n_val:]

print(f'\n📦 Data split:')
print(f'   Train: {len(train_data)} ({len(train_data)*100//n}%)')
print(f'   Val:   {len(val_data)} ({len(val_data)*100//n}%)')
print(f'   Test:  {len(test_data)} ({len(test_data)*100//n}%)')

# Quick stats
verbs = {}
for e in clean_entries:
    v = json.loads(e['output']).get('verb', '?')
    verbs[v] = verbs.get(v, 0) + 1
print(f'\n📋 Verb distribution:')
for k, v in sorted(verbs.items(), key=lambda x: -x[1]):
    print(f'   {k}: {v}')

## Cell 4 — Load Model & Tokenizer

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

print(f'🔧 Loading {MODEL_NAME}...')
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

params = sum(p.numel() for p in model.parameters())
print(f'\n📐 Model parameters: {params:,} ({params/1e6:.0f}M)')
print('✅ Model loaded')

## Cell 5 — Prepare Dataset for Training

In [ ]:
from datasets import Dataset

# Convert to HuggingFace datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

# Tokenization function
def tokenize_function(examples):
    # Tokenize inputs
    model_inputs = tokenizer(
        examples['input'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding='max_length',
    )

    # Tokenize outputs (labels)
    labels = tokenizer(
        examples['output'],
        max_length=MAX_OUTPUT_LENGTH,
        truncation=True,
        padding='max_length',
    )

    # Replace padding token IDs with -100 so they're ignored in loss
    label_ids = labels['input_ids']
    label_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in label_ids
    ]
    model_inputs['labels'] = label_ids
    return model_inputs

print('🔄 Tokenizing datasets...')
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=['input', 'output'])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=['input', 'output'])
tokenized_test = test_dataset.map(tokenize_function, batched=True, remove_columns=['input', 'output'])

# Set format for PyTorch
tokenized_train.set_format('torch')
tokenized_val.set_format('torch')
tokenized_test.set_format('torch')

print(f'✅ Tokenization complete')
print(f'   Train samples: {len(tokenized_train)}')
print(f'   Val samples:   {len(tokenized_val)}')
print(f'   Test samples:  {len(tokenized_test)}')

## Cell 6 — Train

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    label_smoothing_factor=LABEL_SMOOTHING,
    warmup_steps=WARMUP_STEPS,

    # Evaluation & saving
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Generation settings for eval
    predict_with_generate=True,
    generation_max_length=MAX_OUTPUT_LENGTH,

    # Logging
    logging_steps=50,
    report_to='none',       # No wandb/mlflow

    # Performance
    fp16=torch.cuda.is_available(),  # Mixed precision on GPU
    dataloader_num_workers=2,
    save_total_limit=3,     # Keep only 3 best checkpoints
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print('🚀 Starting training...')
print(f'   {len(tokenized_train)} train samples × {EPOCHS} epochs = {len(tokenized_train) * EPOCHS} steps max')
print(f'   Early stopping patience: {PATIENCE} epochs')
print(f'   Eval after every epoch on {len(tokenized_val)} val samples')
print()

train_result = trainer.train()

print(f'\n✅ Training complete!')
print(f'   Total steps: {train_result.global_step}')
print(f'   Final train loss: {train_result.training_loss:.4f}')

## Cell 7 — Evaluate on Test Set (Exact Match Accuracy)

In [ ]:
import json

print('📊 Evaluating on held-out test set...')
print(f'   Test samples: {len(test_data)}')

# Generate predictions
model.eval()
model.to(device)

correct = 0
valid_json = 0
verb_correct = 0
site_correct = 0
total = len(test_data)
errors = []

for i, entry in enumerate(test_data):
    # Tokenize input
    inputs = tokenizer(
        entry['input'],
        return_tensors='pt',
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    ).to(device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_OUTPUT_LENGTH,
            num_beams=4,           # Beam search for better quality
            early_stopping=True,
            no_repeat_ngram_size=0,
        )

    predicted = tokenizer.decode(outputs[0], skip_special_tokens=True)
    expected = entry['output']

    # Check exact match
    if predicted.strip() == expected.strip():
        correct += 1

    # Check valid JSON
    try:
        pred_json = json.loads(predicted)
        exp_json = json.loads(expected)
        valid_json += 1

        # Check verb accuracy
        if pred_json.get('verb') == exp_json.get('verb'):
            verb_correct += 1

        # Check site accuracy
        if pred_json.get('site') == exp_json.get('site'):
            site_correct += 1

    except json.JSONDecodeError:
        if len(errors) < 10:
            errors.append((entry['input'][:60], predicted[:80]))

    # Progress
    if (i + 1) % 50 == 0:
        print(f'   Processed {i+1}/{total}...')

print(f'\n' + '='*50)
print(f'📋 TEST RESULTS ({total} samples)')
print(f'='*50)
print(f'   Valid JSON:     {valid_json}/{total} ({valid_json*100//total}%)')
print(f'   Exact match:    {correct}/{total} ({correct*100//total}%)')
print(f'   Verb accuracy:  {verb_correct}/{total} ({verb_correct*100//total}%)')
print(f'   Site accuracy:  {site_correct}/{total} ({site_correct*100//total}%)')

if errors:
    print(f'\n⚠️ Invalid JSON examples ({len(errors)}):')  
    for inp, pred in errors[:5]:
        print(f'   Input: {inp}')
        print(f'   Output: {pred}')
        print()

## Cell 8 — Interactive Testing

In [ ]:
def predict(prompt):
    """Run a single prompt through the trained model."""
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_OUTPUT_LENGTH,
            num_beams=4,
            early_stopping=True,
        )

    raw = tokenizer.decode(outputs[0], skip_special_tokens=True)

    try:
        parsed = json.loads(raw)
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError:
        print(f'⚠️ Invalid JSON: {raw}')
    return raw

# ============================================================
# TEST PROMPTS — Try your own!
# ============================================================

test_prompts = [
    "Find the cheapest laptop on amazon with 32GB RAM",
    "go to youtube",
    "search for hotels in Paris under $200",
    "compare the MacBook Air and MacBook Pro",
    "book a flight from NYC to London on June 15",
    "add the cheapest Nike shoes to my cart",
    "find me trendy women's dresses on SHEIN this season",
    "if the price is under $500, add it to cart",
]

for prompt in test_prompts:
    print(f'\n📤 "{prompt}"')
    print('📥', end=' ')
    predict(prompt)
    print('-' * 60)

## Cell 9 — Save Final Model

In [ ]:
# Save the best model
FINAL_DIR = './goal_parser_final'
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f'✅ Model saved to {FINAL_DIR}/')

# Check model size
import os
total_size = sum(
    os.path.getsize(os.path.join(FINAL_DIR, f))
    for f in os.listdir(FINAL_DIR)
    if os.path.isfile(os.path.join(FINAL_DIR, f))
)
print(f'📦 Model size: {total_size / 1e6:.1f} MB')

## Cell 10 — Download Model

In [ ]:
# Zip and download the trained model
import shutil
shutil.make_archive('goal_parser_final', 'zip', FINAL_DIR)

from google.colab import files
files.download('goal_parser_final.zip')
print('📥 Download started!')